In [1]:
!pip install -q openai-agents yfinance resend schedule


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.1/856.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 6.3 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["RESEND_API_KEY"] = getpass("Enter Resend API Key: ")

Enter OpenAI API Key: ··········
Enter Resend API Key: ··········


In [7]:
import yfinance as yf

def get_stock_info(ticker):

    stock = yf.Ticker(ticker)

    info = stock.history(period="2d")

    current = info["Close"].iloc[-1]

    previous = info["Close"].iloc[-2]

    change = (current - previous) / previous * 100

    return {

        "ticker": ticker,

        "current": current,

        "previous": previous,

        "change": change

    }

In [6]:
from agents import Agent

price_agent = Agent(

    name="Price Analyst",

    instructions="""
    Analyze stock price movements.
    Explain whether the movement is significant.
    Keep your answer under 50 words.
    """,

    model="gpt-5.5"

)

news_agent = Agent(

    name="News Researcher",

    instructions="""
    Search for possible reasons behind today's stock movement.
    Summarize recent news in under 75 words.
    """,

    model="gpt-5.5"

)

summary_agent = Agent(

    name="Investment Summary",

    instructions="""
    Combine multiple analyses into a professional email.

    Keep the response under 120 words.

    """,

    model="gpt-5.5"

)

In [9]:
from agents import Runner

In [11]:
stock = get_stock_info("MSFT")

price_prompt = f"""
Ticker: {stock['ticker']}

Current Price: {stock['current']}

Previous Close: {stock['previous']}

Percentage Change: {stock['change']:.2f}%

Explain this price movement.
"""

price_result = await Runner.run(
    price_agent,
    price_prompt
)

print(price_result.final_output)

MSFT is up 5.04%, a significant one-day move for a mega-cap stock. This likely reflects strong positive news, earnings, guidance, AI/cloud optimism, or broad market strength. Such a move suggests elevated investor demand, but confirming the catalyst requires checking current news and market context.


In [12]:
news_prompt = f"""
The stock {stock['ticker']} moved {stock['change']:.2f}% today.

Use your knowledge and reasoning to explain possible reasons
behind this movement.
"""

news_result = await Runner.run(

    news_agent,

    news_prompt

)

print(news_result.final_output)

MSFT’s 5.04% move today may be tied to investor reaction to recent Microsoft news and broader tech sentiment. Possible drivers include continued optimism around Azure growth and AI monetization through Copilot/OpenAI integrations, expectations for strong enterprise software demand, or analyst upgrades/target increases. The move could also reflect macro factors such as lower rate expectations boosting megacap tech valuations, alongside broader Nasdaq strength. Check today’s earnings, guidance, regulatory, or AI-related headlines for confirmation.


In [13]:
summary_prompt = f"""

Price Analysis

----------------

{price_result.final_output}

News Analysis

----------------

{news_result.final_output}

Create a professional investment summary suitable for an email.
"""

summary_result = await Runner.run(

    summary_agent,

    summary_prompt

)

print(summary_result.final_output)

Subject: Microsoft (MSFT) Investment Summary

Hi [Name],

Microsoft (MSFT) is up 5.04% today, a notable one-day move for a mega-cap stock and indicative of strong investor demand. The gain may be driven by positive sentiment around Azure growth, AI monetization opportunities through Copilot and OpenAI integrations, continued enterprise software demand, or analyst upgrades.

Broader market factors may also be contributing, including lower rate expectations and strength across the Nasdaq and large-cap technology names. Given the size of the move, it would be prudent to confirm the specific catalyst by reviewing today’s earnings, guidance, regulatory updates, and AI-related headlines.

Best,  
[Your Name]


In [14]:
def send_email(subject, body):

    response = resend.Emails.send({

        "from": "onboarding@resend.dev",

        "to": "youremail@gmail.com",

        "subject": subject,

        "html": f"""
        <html>

        <body style="font-family:Arial">

        <h2>📈 AI Stock Alert</h2>

        <hr>

        <p>{body}</p>

        </body>

        </html>
        """

    })

    return response

In [15]:
subject = f"🚨 {stock['ticker']} Alert ({stock['change']:.2f}%)"

response = send_email(

    subject,

    summary_result.final_output

)

print(response)

{'id': 'f42c0bf1-593e-4742-a72a-3eb6fedbbceb', 'http_headers': {'Date': 'Fri, 26 Jun 2026 17:53:42 GMT', 'Content-Type': 'application/json', 'Content-Length': '45', 'Connection': 'keep-alive', 'ratelimit-limit': '10', 'ratelimit-policy': '10;w=1', 'ratelimit-remaining': '9', 'ratelimit-reset': '1', 'x-resend-daily-quota': '0', 'x-resend-monthly-quota': '31', 'cf-cache-status': 'DYNAMIC', 'Server': 'cloudflare', 'CF-RAY': 'a11e12ae784b3d6a-DEN'}}
